# **Processing MedMCQA dataset**

**The MedMCQA dataset used for test will be the dev dataset**

The official MedMCQA test set does not have publicly available answers. Therefore, is used the Validation Set (Dev Set) as the "test set".

## **MedMCQA Subject Analysis**

This script identifies all unique values in the subject_name field and counts how many questions are associated with each.

In [ ]:
import json
import os
from collections import Counter

# --- CONFIGURATION ---
INPUT_FILE = "../../../Data/MedMCQA/dev.json" 

def analyze_medmcqa_subjects(file_path):
    print(f"Analyzing subjects in: {file_path}")
    
    if not os.path.exists(file_path):
        print(f" Error: File '{file_path}' not found. Please check the path.")
        return

    subject_counts = Counter()
    total_entries = 0

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                
                try:
                    data = json.loads(line)
                    total_entries += 1
                    
                    subject = data.get("subject_name")
                    
                    if subject is None:
                        subject_counts["[No Subject]"] += 1
                    else:
                        subject_counts[subject] += 1
                        
                except json.JSONDecodeError:
                    print(f"Warning: Skipping invalid JSON on line {line_num}")
                    continue

        print("-" * 40)
        print(f"ANALYSIS RESULTS")
        print(f"   Total lines processed: {total_entries}")
        print(f"   Unique subjects found: {len(subject_counts)}")
        print("-" * 40)
        
        # Sort subjects by frequency (highest first)
        sorted_subjects = subject_counts.most_common()
        
        print(f"{'Subject Name':<30} | {'Count':<10}")
        print("-" * 43)
        for subject, count in sorted_subjects:
            print(f"{str(subject):<30} | {count:<10}")
        print("-" * 43)

    except Exception as e:
        print(f" An unexpected error occurred: {e}")

analyze_medmcqa_subjects(INPUT_FILE)

Analyzing subjects in: ../../../Data/MedMCQA/dev.json
----------------------------------------
ANALYSIS RESULTS
   Total lines processed: 4183
   Unique subjects found: 21
----------------------------------------
Subject Name                   | Count     
-------------------------------------------
Dental                         | 1318      
Surgery                        | 369       
Pathology                      | 337       
Medicine                       | 295       
Pharmacology                   | 243       
Anatomy                        | 234       
Pediatrics                     | 234       
Gynaecology & Obstetrics       | 224       
Physiology                     | 171       
Biochemistry                   | 171       
Social & Preventive Medicine   | 129       
Microbiology                   | 122       
Radiology                      | 69        
Forensic Medicine              | 67        
Ophthalmology                  | 58        
ENT                            | 53    

## **MedMCQA Test Set Filtering**
This script filters only the subjects most relevant.

**Selected Subjects:**
- Gynaecology & Obstetrics
- Surgery
- Medicine
- Pediatrics
- Pharmacology
- Radiology
- Orthopaedics
- Skin

In [ ]:
import json
import os

# --- CONFIGURATION ---
INPUT_FILE = "../../../Data/MedMCQA/dev.json" 
OUTPUT_DIR = "intermediate_data"
OUTPUT_FILENAME = "medmcqa_test_filtered"

OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)

# Target Subjects for Evaluation
SELECTED_SUBJECTS = [
    "Gynaecology & Obstetrics",
    "Surgery",
    "Medicine",
    "Pharmacology",
    "Pediatrics",
    "Radiology",
    "Orthopaedics",
    "Skin"
]

def filter_medmcqa_test_to_folder():
    print(f"Starting MedMCQA test/dev filtering...")
    
    if not os.path.exists(OUTPUT_DIR):
        try:
            os.makedirs(OUTPUT_DIR)
            print(f"   - Created directory: '{OUTPUT_DIR}'")
        except OSError as e:
            print(f"Error creating directory: {e}")
            return

    if not os.path.exists(INPUT_FILE):
        print(f"Error: Input file '{INPUT_FILE}' not found.")
        return

    filtered_count = 0
    total_count = 0
    subject_stats = {subject: 0 for subject in SELECTED_SUBJECTS}

    try:
        with open(OUTPUT_PATH, 'w', encoding='utf-8') as outfile:
            with open(INPUT_FILE, 'r', encoding='utf-8') as infile:
                for line in infile:
                    line = line.strip()
                    if not line:
                        continue
                    
                    total_count += 1
                    try:
                        data = json.loads(line)
                        subject = data.get("subject_name")

                        if subject in SELECTED_SUBJECTS:
                            json.dump(data, outfile, ensure_ascii=False)
                            outfile.write('\n')
                            filtered_count += 1
                            subject_stats[subject] += 1

                    except json.JSONDecodeError:
                        continue

        print("-" * 40)
        print(f"FILTERING COMPLETE")
        print(f"   Total entries scanned:  {total_count}")
        print(f"   Filtered entries saved: {filtered_count}")
        print(f"   Destination:           {OUTPUT_PATH}")
        print("-" * 40)
        
        print("Breakdown by Clinical Subject:")
        for sub, count in subject_stats.items():
            print(f"   - {sub:<25}: {count}")
        print("-" * 40)

    except Exception as e:
        print(f"An unexpected error occurred: {e}")

filter_medmcqa_test_to_folder()

Starting MedMCQA test/dev filtering...
----------------------------------------
FILTERING COMPLETE
   Total entries scanned:  4183
   Filtered entries saved: 1471
   Destination:           intermediate_data\medmcqa_test_filtered.json
----------------------------------------
Breakdown by Clinical Subject:
   - Gynaecology & Obstetrics : 224
   - Surgery                  : 369
   - Medicine                 : 295
   - Pharmacology             : 243
   - Pediatrics               : 234
   - Radiology                : 69
   - Orthopaedics             : 20
   - Skin                     : 17
----------------------------------------


## **MedMCQA Test Set formatting**
This script converts the filtered MedMCQA dataset (which uses opa, opb, cop keys) into the standardized instruction format (instruction, input, output).

In [ ]:
import json
import os

# --- CONFIGURATION ---
INPUT_FILE = "intermediate_data/medmcqa_test_filtered.json"
OUTPUT_FILE = "medmcqa_test.jsonl"

SYSTEM_INSTRUCTION = (
    "You are a clinical decision support assistant taking a medical certification exam. "
    "Read the question and options carefully. Output ONLY the letter (A, B, C, or D) "
    "corresponding to the correct answer."
)

def convert_to_medqa_format():
    print(f"Converting '{INPUT_FILE}' to MedQA format...")

    if not os.path.exists(INPUT_FILE):
        print(f"Error: File '{INPUT_FILE}' not found.")
        return

    formatted_entries = []
    processed_count = 0
    
    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                
                try:
                    data = json.loads(line)
                    
                    question = data.get('question', '').strip()

                    topic_name = data.get('subject_name', 'Unknown')
                    
                    options_block = (
                        f"A. {data.get('opa', '').strip()}\n"
                        f"B. {data.get('opb', '').strip()}\n"
                        f"C. {data.get('opc', '').strip()}\n"
                        f"D. {data.get('opd', '').strip()}"
                    )
                    
                    formatted_input = f"###Question: {question}\n###Options:\n{options_block}"

                    # Map 'cop' (1,2,3,4) to Letter (A,B,C,D)
                    cop = data.get('cop') 
                    correct_letter = ""
                    
                    if cop == 1: correct_letter = "A"
                    elif cop == 2: correct_letter = "B"
                    elif cop == 3: correct_letter = "C"
                    elif cop == 4: correct_letter = "D"
                    
                    entry = {
                        "instruction": SYSTEM_INSTRUCTION,
                        "input": formatted_input,
                        "output": correct_letter,
                        "topic": topic_name
                    }
                    
                    formatted_entries.append(entry)
                    processed_count += 1

                except json.JSONDecodeError:
                    continue

        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            for entry in formatted_entries:
                f.write(json.dumps(entry, ensure_ascii=False) + '\n')

        print("-" * 40)
        print(f"CONVERSION COMPLETE")
        print(f"   - Total processed: {processed_count}")
        print(f"   - File created:    {OUTPUT_FILE}")
        print("-" * 40)
        
        empty_outputs = sum(1 for e in formatted_entries if e['output'] == "")
        if empty_outputs > 0:
            print(f" Warning: {empty_outputs} entries have no answer (empty 'output').")
            print("    This is expected if you used the original 'test.json' (no 'cop').")

    except Exception as e:
        print(f" Error: {e}")

convert_to_medqa_format()

Converting 'intermediate_data/medmcqa_test_filtered.json' to MedQA format...
----------------------------------------
CONVERSION COMPLETE
   - Total processed: 1471
   - File created:    medmcqa_test.jsonl
----------------------------------------


## **Few-shot prompting**

Choosed manually 5 examples from the train dataset with all the letters being the correct option (A,B,C,D) 